## Raw Data Transformations for Power BI

Transforms wide-format FAO raw tables into long format for Power BI visualizations. Original raw tables are kept untouched — new `_long` tables are created as duplicates.

Tables transformed:
- raw.Area_protected_by_law_to_reamin_as_forest
- raw.Forest_area_change
- raw.Naturaly_regenerating_and_primary_forest
- raw.Planted_forest
- raw.forest_purpose
- raw.Forest_change_reason

Notes:
- Country codes added via join with 'raw.Forest_Policy_Legislation' (iso3)
- UK (GBR) mapped manually — not in Forest_Policy_Legislation
- Original wide-format tables preserved for Python statistics notebooks

In [20]:
-- Check column names for all tables that might be in wide format
SELECT TABLE_NAME, COLUMN_NAME
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'raw'
    AND TABLE_NAME IN (
        'Forest_area_change',
        'Naturaly_regenerating_and_primary_forest',
        'Planted_forest',
        'Forest_Policy_Legislation',
        'Forest_change_reason',
        'forest_purpose'
    )
ORDER BY TABLE_NAME, ORDINAL_POSITION;

(111 rows affected)

TABLE_NAME                               | COLUMN_NAME                                        
-----------------------------------------+----------------------------------------------------
Forest_area_change                       | Column1                                            
Forest_area_change                       | …of which natural expansion (1 000 ha/year)        
Forest_area_change                       | …of which natural expansion (1 000 ha/year)_1      
Forest_area_change                       | …of which natural expansion (1 000 ha/year)_2      
Forest_area_change                       | …of which natural expansion (1 000 ha/year)_3      
Forest_area_change                       | …of which natural expansion (1 000 ha/year)_4      
Forest_change_reason                     | regions                                            
Forest_change_reason                     | subregions                                         
Forest_change_reason         

In [ ]:
-- -- TABLE 1: Area protected by law
-- -- Wide format (years as columns)
-- -- Years: 1990, 2000, 2010, 2015, 2020
SELECT 
    fpl.iso3                AS country_code,
    src.country_name,
    src.year,
    src.protected_area_1000ha
INTO raw.Area_protected_long
FROM (
    SELECT [Column1] AS country_name, 1990 AS year, 
           [Area of permanent forest estate (1 000 ha)]   AS protected_area_1000ha 
    FROM raw.Area_protected_by_law_to_reamin_as_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2000, [Area of permanent forest estate (1 000 ha)_1] 
    FROM raw.Area_protected_by_law_to_reamin_as_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2010, [Area of permanent forest estate (1 000 ha)_2] 
    FROM raw.Area_protected_by_law_to_reamin_as_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2015, [Area of permanent forest estate (1 000 ha)_3] 
    FROM raw.Area_protected_by_law_to_reamin_as_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2020, [Area of permanent forest estate (1 000 ha)_4] 
    FROM raw.Area_protected_by_law_to_reamin_as_forest WHERE [Column1] NOT LIKE '199%'
) src
JOIN raw.Forest_Policy_Legislation fpl ON fpl.name = src.country_name
WHERE src.country_name NOT LIKE '%FRA%'
    AND src.country_name IS NOT NULL
    AND src.country_name != '';

-- Fix UK manually
UPDATE raw.Area_protected_long
SET country_code = 'GBR'
WHERE country_name = 'United Kingdom of Great Britain and Northern Ireland';

-- Verify
SELECT TOP 5 * FROM raw.Area_protected_long ORDER BY country_code, year;

(5 rows affected)

country_name | year | protected_area_1000ha | country_code
-------------+------+-----------------------+-------------
Aruba        | 1990 | NULL                  | ABW         
Aruba        | 2000 | NULL                  | ABW         
Aruba        | 2010 | NULL                  | ABW         
Aruba        | 2015 | NULL                  | ABW         
Aruba        | 2020 | NULL                  | ABW         
(5 rows)

Total execution time: 00:00:01.003

In [ ]:
-- TABLE 2: Natural forest expansion
-- Periods: 1990-2000, 2000-2010, 2010-2015, 2015-2020, 2020-2025
SELECT 
    fpl.iso3                    AS country_code,
    src.country_name,
    src.period,
    src.natural_expansion_1000ha
INTO raw.Forest_area_change_long
FROM (
    SELECT [Column1] AS country_name, '1990-2000' AS period, 
           […of which natural expansion (1 000 ha/year)]   AS natural_expansion_1000ha 
    FROM raw.Forest_area_change WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], '2000-2010', […of which natural expansion (1 000 ha/year)_1] 
    FROM raw.Forest_area_change WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], '2010-2015', […of which natural expansion (1 000 ha/year)_2] 
    FROM raw.Forest_area_change WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], '2015-2020', […of which natural expansion (1 000 ha/year)_3] 
    FROM raw.Forest_area_change WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], '2020-2025', […of which natural expansion (1 000 ha/year)_4] 
    FROM raw.Forest_area_change WHERE [Column1] NOT LIKE '199%'
) src
JOIN raw.Forest_Policy_Legislation fpl ON fpl.name = src.country_name;

-- Fix UK manually
UPDATE raw.Forest_area_change_long
SET country_code = 'GBR'
WHERE country_name = 'United Kingdom of Great Britain and Northern Ireland';

-- Verify
SELECT TOP 5 * FROM raw.Forest_area_change_long ORDER BY country_code, period;

(1175 rows affected)
(0 rows affected)
(5 rows affected)

country_code | country_name | period    | natural_expansion_1000ha
-------------+--------------+-----------+-------------------------
ABW          | Aruba        | 1990-2000 |                         
ABW          | Aruba        | 2000-2010 |                         
ABW          | Aruba        | 2010-2015 |                         
ABW          | Aruba        | 2015-2020 |                         
ABW          | Aruba        | 2020-2025 |                         
(5 rows)

Total execution time: 00:00:00.073

In [ ]:
-- TABLE 3: Naturally regenerating and primary forest
-- Years: 1990, 2000, 2010, 2015, 2020, 2025
SELECT 
    fpl.iso3                            AS country_code,
    src.country_name,
    src.year,
    src.naturally_regenerating_1000ha,
    src.primary_forest_1000ha
INTO raw.Naturaly_regenerating_long
FROM (
    SELECT [Column1] AS country_name, 1990 AS year,
           [Naturally regenerating forest (1 000 ha)]   AS naturally_regenerating_1000ha,
           […of which primary forest (1 000 ha)]         AS primary_forest_1000ha
    FROM raw.Naturaly_regenerating_and_primary_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2000,
           [Naturally regenerating forest (1 000 ha)_1],
           […of which primary forest (1 000 ha)_1]
    FROM raw.Naturaly_regenerating_and_primary_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2010,
           [Naturally regenerating forest (1 000 ha)_2],
           […of which primary forest (1 000 ha)_2]
    FROM raw.Naturaly_regenerating_and_primary_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2015,
           [Naturally regenerating forest (1 000 ha)_3],
           […of which primary forest (1 000 ha)_3]
    FROM raw.Naturaly_regenerating_and_primary_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2020,
           [Naturally regenerating forest (1 000 ha)_4],
           […of which primary forest (1 000 ha)_4]
    FROM raw.Naturaly_regenerating_and_primary_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2025,
           [Naturally regenerating forest (1 000 ha)_5],
           […of which primary forest (1 000 ha)_5]
    FROM raw.Naturaly_regenerating_and_primary_forest WHERE [Column1] NOT LIKE '199%'
) src
JOIN raw.Forest_Policy_Legislation fpl ON fpl.name = src.country_name;

-- Fix UK manually
UPDATE raw.Naturaly_regenerating_long
SET country_code = 'GBR'
WHERE country_name = 'United Kingdom of Great Britain and Northern Ireland';

-- Verify
SELECT TOP 5 * FROM raw.Naturaly_regenerating_long ORDER BY country_code, year;

(1410 rows affected)
(0 rows affected)
(5 rows affected)

country_code | country_name | year | naturally_regenerating_1000ha | primary_forest_1000ha
-------------+--------------+------+-------------------------------+----------------------
ABW          | Aruba        | 1990 | 0,49                          | NULL                 
ABW          | Aruba        | 2000 | 0,49                          | NULL                 
ABW          | Aruba        | 2010 | 0,49                          | NULL                 
ABW          | Aruba        | 2015 | 0,49                          | NULL                 
ABW          | Aruba        | 2020 | 0,49                          | NULL                 
(5 rows)

Total execution time: 00:00:00.079

In [ ]:
-- TABLE 4: Planted forest growing stock
-- Years: 1990, 2000, 2010, 2015, 2020, 2025
SELECT 
    fpl.iso3                        AS country_code,
    src.country_name,
    src.year,
    src.planted_forest_million_m3
INTO raw.Planted_forest_long
FROM (
    SELECT [Column1] AS country_name, 1990 AS year,
           [Planted forest (million m³ over bark)]   AS planted_forest_million_m3
    FROM raw.Planted_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2000, [Planted forest (million m³ over bark)_1]
    FROM raw.Planted_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2010, [Planted forest (million m³ over bark)_2]
    FROM raw.Planted_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2015, [Planted forest (million m³ over bark)_3]
    FROM raw.Planted_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2020, [Planted forest (million m³ over bark)_4]
    FROM raw.Planted_forest WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2025, [Planted forest (million m³ over bark)_5]
    FROM raw.Planted_forest WHERE [Column1] NOT LIKE '199%'
) src
JOIN raw.Forest_Policy_Legislation fpl ON fpl.name = src.country_name;

-- Fix UK manually
UPDATE raw.Planted_forest_long
SET country_code = 'GBR'
WHERE country_name = 'United Kingdom of Great Britain and Northern Ireland';

-- Verify
SELECT TOP 5 * FROM raw.Planted_forest_long ORDER BY country_code, year;

(1410 rows affected)
(0 rows affected)
(5 rows affected)

country_code | country_name | year | planted_forest_million_m3
-------------+--------------+------+--------------------------
ABW          | Aruba        | 1990 | 0                        
ABW          | Aruba        | 2000 | 0                        
ABW          | Aruba        | 2010 | 0                        
ABW          | Aruba        | 2015 | 0                        
ABW          | Aruba        | 2020 | 0                        
(5 rows)

Total execution time: 00:00:00.078

In [ ]:
-- TABLE 5: Forest purpose designation
-- Years: 1990, 2000, 2010, 2015, 2020, 2025
-- 9 purpose categories per row
SELECT 
    fpl.iso3                        AS country_code,
    src.country_name,
    src.year,
    src.production_1000ha,
    src.protection_1000ha,
    src.biodiversity_1000ha,
    src.social_services_1000ha,
    src.multiple_use_1000ha,
    src.other_1000ha,
    src.no_designation_1000ha,
    src.unknown_1000ha,
    src.total_forest_area_1000ha
INTO raw.forest_purpose_long
FROM (
    SELECT [Column1] AS country_name, 1990 AS year,
           [Production (1 000 ha)], [Protection of soil and water (1 000 ha)],
           [Conservation of biodiversity (1 000 ha)], [Social Services (1 000 ha)],
           [Multiple use (1 000 ha)], [Other (specify in comments) (1 000 ha)],
           [No designation (1 000 ha)], [Unknown (1 000 ha)],
           [Total forest area (1 000 ha)]
    FROM raw.forest_purpose WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2000,
           [Production (1 000 ha)_1], [Protection of soil and water (1 000 ha)_1],
           [Conservation of biodiversity (1 000 ha)_1], [Social Services (1 000 ha)_1],
           [Multiple use (1 000 ha)_1], [Other (specify in comments) (1 000 ha)_1],
           [No designation (1 000 ha)_1], [Unknown (1 000 ha)_1],
           [Total forest area (1 000 ha)_1]
    FROM raw.forest_purpose WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2010,
           [Production (1 000 ha)_2], [Protection of soil and water (1 000 ha)_2],
           [Conservation of biodiversity (1 000 ha)_2], [Social Services (1 000 ha)_2],
           [Multiple use (1 000 ha)_2], [Other (specify in comments) (1 000 ha)_2],
           [No designation (1 000 ha)_2], [Unknown (1 000 ha)_2],
           [Total forest area (1 000 ha)_2]
    FROM raw.forest_purpose WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2015,
           [Production (1 000 ha)_3], [Protection of soil and water (1 000 ha)_3],
           [Conservation of biodiversity (1 000 ha)_3], [Social Services (1 000 ha)_3],
           [Multiple use (1 000 ha)_3], [Other (specify in comments) (1 000 ha)_3],
           [No designation (1 000 ha)_3], [Unknown (1 000 ha)_3],
           [Total forest area (1 000 ha)_3]
    FROM raw.forest_purpose WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2020,
           [Production (1 000 ha)_4], [Protection of soil and water (1 000 ha)_4],
           [Conservation of biodiversity (1 000 ha)_4], [Social Services (1 000 ha)_4],
           [Multiple use (1 000 ha)_4], [Other (specify in comments) (1 000 ha)_4],
           [No designation (1 000 ha)_4], [Unknown (1 000 ha)_4],
           [Total forest area (1 000 ha)_4]
    FROM raw.forest_purpose WHERE [Column1] NOT LIKE '199%'
    UNION ALL
    SELECT [Column1], 2025,
           [Production (1 000 ha)_5], [Protection of soil and water (1 000 ha)_5],
           [Conservation of biodiversity (1 000 ha)_5], [Social Services (1 000 ha)_5],
           [Multiple use (1 000 ha)_5], [Other (specify in comments) (1 000 ha)_5],
           [No designation (1 000 ha)_5], [Unknown (1 000 ha)_5],
           [Total forest area (1 000 ha)_5]
    FROM raw.forest_purpose WHERE [Column1] NOT LIKE '199%'
) src (country_name, year, production_1000ha, protection_1000ha, biodiversity_1000ha,
       social_services_1000ha, multiple_use_1000ha, other_1000ha,
       no_designation_1000ha, unknown_1000ha, total_forest_area_1000ha)
JOIN raw.Forest_Policy_Legislation fpl ON fpl.name = src.country_name;

-- Fix UK manually
UPDATE raw.forest_purpose_long
SET country_code = 'GBR'
WHERE country_name = 'United Kingdom of Great Britain and Northern Ireland';

-- Verify
SELECT TOP 5 * FROM raw.forest_purpose_long ORDER BY country_code, year;

(1332 rows affected)
(0 rows affected)
(5 rows affected)

country_code | country_name | year | production_1000ha | protection_1000ha | biodiversity_1000ha | social_services_1000ha | multiple_use_1000ha | other_1000ha | no_designation_1000ha | unknown_1000ha | total_forest_area_1000ha
-------------+--------------+------+-------------------+-------------------+---------------------+------------------------+---------------------+--------------+-----------------------+----------------+-------------------------
ABW          | Aruba        | 1990 | 0                 | 0                 | 0                   | 0                      | 0,04                | NULL         | NULL                  | 0,45           | 0,49                    
ABW          | Aruba        | 2000 | 0                 | 0                 | 0                   | 0                      | 0,04                | NULL         | NULL                  | 0,45           | 0,49                    
ABW          | Aruba        | 2010

In [ ]:
-- TABLE 6: Forest disturbances (fire, insects, diseases)
-- Already in long format with iso3 — just create clean copy with readable column names
-- Units: 1 000 ha
SELECT 
    iso3                AS country_code,
    name                AS country_name,
    year,
    [5a_insect]         AS insect_1000ha,
    [5a_diseases]       AS diseases_1000ha,
    [5a_weather]        AS weather_1000ha,
    [5a_other]          AS other_1000ha,
    [5b_fire_land]      AS fire_land_1000ha,
    [5b_fire_forest]    AS fire_forest_1000ha
INTO raw.Forest_change_reason_long
FROM raw.Forest_change_reason;

-- Verify
SELECT TOP 5 * FROM raw.Forest_change_reason_long ORDER BY country_code, year;

(5328 rows affected)
(5 rows affected)

country_code | country_name | year | insect_1000ha | diseases_1000ha | weather_1000ha | other_1000ha | fire_land_1000ha | fire_forest_1000ha
-------------+--------------+------+---------------+-----------------+----------------+--------------+------------------+-------------------
ABW          | Aruba        | 2000 | NULL          | NULL            |                |              | 0                | 0                 
ABW          | Aruba        | 2001 | NULL          | NULL            |                |              | 0                | 0                 
ABW          | Aruba        | 2002 | NULL          | NULL            |                |              | 0                | 0                 
ABW          | Aruba        | 2003 | NULL          | NULL            |                |              | 0                | 0                 
ABW          | Aruba        | 2004 | NULL          | NULL            |                |              | 0          

In [21]:
-- Verify all long tables exist and have rows
SELECT 'Area_protected_long'       AS table_name, COUNT(*) AS rows FROM raw.Area_protected_long
UNION ALL
SELECT 'Forest_area_change_long',  COUNT(*) FROM raw.Forest_area_change_long
UNION ALL
SELECT 'Naturaly_regenerating_long', COUNT(*) FROM raw.Naturaly_regenerating_long
UNION ALL
SELECT 'Planted_forest_long',      COUNT(*) FROM raw.Planted_forest_long
UNION ALL
SELECT 'forest_purpose_long',      COUNT(*) FROM raw.forest_purpose_long
UNION ALL
SELECT 'Forest_change_reason_long', COUNT(*) FROM raw.Forest_change_reason_long;

(6 rows affected)

table_name                 | rows
---------------------------+-----
Area_protected_long        | 1175
Forest_area_change_long    | 1175
Naturaly_regenerating_long | 1410
Planted_forest_long        | 1410
forest_purpose_long        | 1332
Forest_change_reason_long  | 5328
(6 rows)

Total execution time: 00:00:00.143